# 21 · Graph Neural Networks and Geometric Deep Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anuxlab/deep-learning-systems/blob/main/notebooks/21-gnn/21_gnn.ipynb)
[![Run on GitHub Actions](https://img.shields.io/badge/run-GitHub%20Actions-2088FF?logo=githubactions&logoColor=white)](https://github.com/anuxlab/deep-learning-systems/actions/workflows/nb-21-gnn.yml)

| | |
|---|---|
| **Wiki chapter** | [Module 21](https://github.com/anuxlab/deep-learning-systems/wiki/Module-21) |
| **Level** | Advanced |
| **Typical CPU runtime** | ~1 min (GitHub free runner, no GPU) |
| **Prerequisites** | Modules 01, 07 |

## What you will do

- Work with adjacency, degree and Laplacian matrices; verify spectral properties and do spectral clustering.
- Implement GCN, GraphSAGE, GAT and GIN layers from scratch (dense form) and test permutation equivariance.
- Train a semi-supervised node classifier on Zachary's karate club with only 2 labelled nodes per class.
- Demonstrate GIN's greater expressiveness over mean aggregation and observe over-smoothing with depth.

> **How to use this notebook.** Read each markdown cell, *predict* what the next code cell will print, then run it.
> Every section ends with `check(...)` assertions: if one fails, something in your understanding (or the code) is off.
> Figures are saved to `figures/` and a machine-readable summary to `metrics.json` next to this notebook.

## 0 · Setup and reproducibility

We import shared helpers (`show` saves each figure to `figures/`, `check` asserts and logs, `timer` logs wall-clock time), fix all random seeds and print an environment report so every run is traceable (see the wiki's *Implementation and Research Standards*).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "_common"))   # shared helpers: notebooks/_common/dlsys.py
from dlsys import *                                       # np, plt, torch, seed_everything, show, check, timer, ...

SEED = seed_everything(42)
ENV = env_report()

## 1 · Graphs as matrices
A graph $G=(V,E)$ with $n$ nodes has adjacency $A\in\{0,1\}^{n\times n}$, degree matrix $D=\mathrm{diag}(A\mathbf 1)$, Laplacian $L=D-A$ and symmetric normalised Laplacian $\mathcal L=I-D^{-1/2}AD^{-1/2}$. Facts: $L$ is PSD; $x^\top Lx=\sum_{(i,j)\in E}(x_i-x_j)^2$ (a *smoothness* measure); the multiplicity of eigenvalue 0 equals the number of connected components; the second eigenvector (Fiedler) splits the graph. We use **Zachary's karate club** (34 members who split into two factions) bundled with `networkx` — no download.

In [ ]:
import networkx as nx
G_ = nx.karate_club_graph(); n = G_.number_of_nodes(); A = nx.to_numpy_array(G_, weight=None); y = np.array([0 if G_.nodes[i]["club"] == "Mr. Hi" else 1 for i in range(n)])
deg = A.sum(1); Dm = np.diag(deg); Lap = Dm - A; Dm12 = np.diag(deg ** -.5); Ln = np.eye(n) - Dm12 @ A @ Dm12
print(f"nodes {n}, edges {int(A.sum()/2)}, degree range {deg.min():.0f}-{deg.max():.0f}, class balance {np.bincount(y)}")
ev, U = np.linalg.eigh(Lap); evn = np.linalg.eigvalsh(Ln)
check(ev.min() > -1e-9 and abs(ev[0]) < 1e-9, "Laplacian is PSD with a zero eigenvalue (connected graph ⇒ exactly one)")
check(ev[1] > 1e-6, "second eigenvalue > 0 ⇒ the graph is connected (algebraic connectivity %.3f)" % ev[1])
check(evn.max() <= 2 + 1e-9, "normalised Laplacian eigenvalues lie in [0, 2]")
xr = np.random.default_rng(0).normal(size=n); check(np.isclose(xr @ Lap @ xr, sum((xr[i] - xr[j]) ** 2 for i, j in G_.edges())), "xᵀLx = Σ_edges (x_i − x_j)²")
fied = U[:, 1]; pred = (fied > 0).astype(int); acc = max((pred == y).mean(), (pred != y).mean()); print(f"spectral clustering with the Fiedler vector recovers the factions with accuracy {acc:.3f}")
check(acc > .9, "Fiedler-vector spectral clustering recovers the real split (≥ 90%)")
pos = nx.spring_layout(G_, seed=3); fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
nx.draw(G_, pos, node_color=y, cmap="coolwarm", node_size=120, ax=ax[0], with_labels=False); ax[0].set_title("True factions")
nx.draw(G_, pos, node_color=fied, cmap="coolwarm", node_size=120, ax=ax[1], with_labels=False); ax[1].set_title("Fiedler vector (2nd Laplacian eigenvector)"); show("karate_spectral"); METRICS["spectral_acc"] = acc

## 2 · Message passing layers (dense implementation)
All GNN layers share the pattern **aggregate neighbours → combine with self → transform**. With features $H\in\mathbb R^{n\times d}$:

- **GCN:** $H'=\sigma(\hat AHW)$, $\hat A=\tilde D^{-1/2}(A+I)\tilde D^{-1/2}$.
- **GraphSAGE-mean:** $H'=\sigma([H\,\|\,D^{-1}AH]W)$.
- **GAT:** attention coefficients $\alpha_{ij}=\mathrm{softmax}_{j\in\mathcal N(i)}\,\mathrm{LeakyReLU}(a^\top[Wh_i\|Wh_j])$, $h_i'=\sigma(\sum_j\alpha_{ij}Wh_j)$.
- **GIN:** $h_i'=\mathrm{MLP}\big((1+\epsilon)h_i+\sum_{j\in\mathcal N(i)}h_j\big)$ — *sum* aggregation.

We write them with dense $n\times n$ matrices (fine for small graphs), and test **permutation equivariance**: relabelling the nodes by a permutation matrix $P$ must permute outputs identically, $f(PAP^\top,PH)=P f(A,H)$.

In [ ]:
def norm_adj(A): At = A + np.eye(len(A)); d = At.sum(1) ** -.5; return d[:, None] * At * d[None]
class GCN(nn.Module):
    def __init__(s, i, o): super().__init__(); s.W = nn.Linear(i, o)
    def forward(s, A, H): return s.W(A_hat(A) @ H)
class SAGE(nn.Module):
    def __init__(s, i, o): super().__init__(); s.W = nn.Linear(2 * i, o)
    def forward(s, A, H): return s.W(torch.cat([H, (A / A.sum(1, keepdim=True).clamp(min=1)) @ H], 1))
class GAT(nn.Module):
    def __init__(s, i, o): super().__init__(); s.W = nn.Linear(i, o, bias=False); s.a_src = nn.Parameter(torch.randn(o) * .1); s.a_dst = nn.Parameter(torch.randn(o) * .1)
    def forward(s, A, H):
        Wh = s.W(H); e = F.leaky_relu((Wh @ s.a_dst)[:, None] + (Wh @ s.a_src)[None], .2); mask = (A + torch.eye(len(A))) > 0
        att = F.softmax(e.masked_fill(~mask, float("-inf")), 1); s.last_att = att; return att @ Wh
class GIN(nn.Module):
    def __init__(s, i, o): super().__init__(); s.eps = nn.Parameter(torch.zeros(1)); s.mlp = nn.Sequential(nn.Linear(i, o), nn.ReLU(), nn.Linear(o, o))
    def forward(s, A, H): return s.mlp((1 + s.eps) * H + A @ H)
def A_hat(A): At = A + torch.eye(len(A)); d = At.sum(1) ** -.5; return d[:, None] * At * d[None]
At_ = torch.tensor(A, dtype=torch.float32); torch.manual_seed(0); Hf = torch.randn(n, 6); P = torch.eye(n)[torch.randperm(n)]
for name, L in [("GCN", GCN(6, 5)), ("GraphSAGE", SAGE(6, 5)), ("GAT", GAT(6, 5)), ("GIN", GIN(6, 5))]:
    out = L(At_, Hf); out_p = L(P @ At_ @ P.T, P @ Hf); check(torch.allclose(P @ out, out_p, atol=1e-5), f"{name}: permutation equivariant  f(PAPᵀ, PH) = P f(A, H)")
Rd = lambda H: H.sum(0); check(torch.allclose(Rd(Hf), Rd(P @ Hf), atol=1e-5), "sum read-out is permutation INVARIANT (needed for graph-level tasks)")
check(torch.allclose(A_hat(At_).sum(), torch.tensor(float(norm_adj(A).sum())), atol=1e-4), "dense GCN normalisation matches the reference")

## 3 · Semi-supervised node classification (2 labels per class!)
Only **4 labelled nodes** (the two club leaders plus one other member per class); the GCN must label the remaining 30 using graph structure and (here) identity features $H=I$. Features carry no information, so everything the model learns comes from message passing — a clean demonstration that *structure is signal*. We train GCN, GraphSAGE, GAT and GIN (2 layers each) and compare test accuracy over several seeds. A feature-only baseline (MLP on $H=I$ with the same labels) cannot generalise at all.

In [ ]:
Hid = torch.eye(n); yt = torch.tensor(y); rng = np.random.default_rng(0)
lab = np.array([0, 33, int(np.where((y == 0) & (deg == np.sort(deg[y == 0])[-3]))[0][0]), int(np.where((y == 1) & (deg == np.sort(deg[y == 1])[-3]))[0][0])]); mask = torch.zeros(n, dtype=torch.bool); mask[lab] = True
print("labelled nodes:", lab, "classes:", y[lab])
class Net(nn.Module):
    def __init__(s, kind, h=16): 
        super().__init__(); L = {"GCN": GCN, "GraphSAGE": SAGE, "GAT": GAT, "GIN": GIN}[kind] if kind != "MLP" else None; s.kind = kind
        s.l1 = L(n, h) if L else nn.Linear(n, h); s.l2 = L(h, 2) if L else nn.Linear(h, 2); s.drop = nn.Dropout(.3)
    def forward(s, A, H): h = F.relu(s.l1(A, H) if s.kind != "MLP" else s.l1(H)); h = s.drop(h); return s.l2(A, h) if s.kind != "MLP" else s.l2(h)
def run_node(kind, seed, epochs=200):
    torch.manual_seed(seed); m = Net(kind); o = torch.optim.Adam(m.parameters(), 1e-2, weight_decay=5e-4)
    for ep in range(epochs): m.train(); out = m(At_, Hid); loss = F.cross_entropy(out[mask], yt[mask]); o.zero_grad(); loss.backward(); o.step()
    m.eval(); pred = m(At_, Hid).argmax(1); return (pred[~mask] == yt[~mask]).float().mean().item(), m
res = {}
with timer("node_classification"):
    for kind in ["MLP", "GCN", "GraphSAGE", "GAT", "GIN"]: accs = [run_node(kind, s)[0] for s in range(5)]; res[kind] = (np.mean(accs), np.std(accs)); print(f"{kind:10s} test acc over 5 seeds: {res[kind][0]:.3f} ± {res[kind][1]:.3f}")
plt.bar(res.keys(), [v[0] for v in res.values()], yerr=[v[1] for v in res.values()]); plt.ylabel("accuracy on 30 unlabelled nodes"); plt.ylim(0, 1.05); plt.title("Karate club: 4 labels, structure-only features"); show("node_classification")
check(res["GCN"][0] > .85, "GCN labels ≥ 85% of unlabelled nodes from only 4 labels"); check(res["GCN"][0] > res["MLP"][0] + .2, "graph structure helps hugely vs a feature-only MLP")
METRICS["node_acc"] = {k: dict(mean=v[0], std=v[1]) for k, v in res.items()}
_, gcn = run_node("GCN", 0); 
with torch.no_grad(): emb = F.relu(gcn.l1(At_, Hid)).numpy()
from sklearn.decomposition import PCA
e2 = PCA(2).fit_transform(emb); plt.scatter(e2[:, 0], e2[:, 1], c=y, cmap="coolwarm", s=60, edgecolor="k"); plt.scatter(*e2[lab].T, marker="*", s=300, c="gold", edgecolor="k"); plt.title("GCN hidden embedding (★ = labelled)"); show("gcn_embedding")

## 4 · Expressiveness: sum vs mean vs max aggregation
Node A has neighbour features $\{1,1\}$; node B has $\{1\}$; as *multisets* they differ, but **mean** (and max) aggregation returns the same value 1 for both — they cannot tell A from B. **Sum** gives 2 vs 1. This is the core of Xu et al.'s result: GIN with sum aggregation + injective MLP is as discriminative as the 1-WL colour-refinement test, while GCN/SAGE-mean are strictly weaker. We build two small graphs and show it concretely, plus the classic pair the 1-WL test *cannot* separate (6-cycle vs two triangles) for which **no** such message-passing GNN can.

In [ ]:
def agg_demo():
    Hn = torch.tensor([[1.], [1.], [1.], [1.], [1.]]); Ag = torch.zeros(5, 5)          # node 0 has neighbours {1,2}; node 3 has neighbour {4}
    for i, j in [(0, 1), (0, 2), (3, 4)]: Ag[i, j] = Ag[j, i] = 1
    s = Ag @ Hn; m = s / Ag.sum(1, keepdim=True).clamp(min=1); return s[[0, 3]].squeeze(), m[[0, 3]].squeeze()
s, m = agg_demo(); print("neighbour SUM  of node A vs node B:", s.tolist(), "| MEAN:", m.tolist())
check(s[0] != s[1] and m[0] == m[1], "sum separates the two multisets; mean cannot")
def wl_colors(A, iters=3):
    c = [0] * len(A)
    for _ in range(iters): c = [hash((c[i], tuple(sorted(c[j] for j in np.where(A[i] > 0)[0])))) for i in range(len(A))]
    return sorted(c)
C6 = np.zeros((6, 6)); 
for i in range(6): C6[i, (i + 1) % 6] = C6[(i + 1) % 6, i] = 1
TT = np.zeros((6, 6))
for tri in [(0, 1, 2), (3, 4, 5)]:
    for a_, b_ in [(0, 1), (1, 2), (0, 2)]: TT[tri[a_], tri[b_]] = TT[tri[b_], tri[a_]] = 1
check(wl_colors(C6) == wl_colors(TT), "1-WL cannot distinguish a 6-cycle from two triangles (all nodes have degree 2) — a hard limit of message passing")
gin = GIN(1, 4); x1 = torch.ones(6, 1); o1, o2 = gin(torch.tensor(C6, dtype=torch.float32), x1).sum(0), gin(torch.tensor(TT, dtype=torch.float32), x1).sum(0); check(torch.allclose(o1, o2, atol=1e-6), "…and consequently a GIN gives identical graph embeddings for both")

## 5 · Over-smoothing
Repeated GCN propagation $\hat A^kX$ converges (for a connected non-bipartite graph) to the dominant eigenvector direction, so all node features become indistinguishable. We measure the **Dirichlet energy**-like quantity: the mean pairwise distance between node representations after $k$ propagation steps, normalised, and the accuracy of a classifier on the propagated features. Deep GCNs lose information — remedies: residual/skip connections, jumping knowledge, normalisation, fewer layers.

In [ ]:
Ah = norm_adj(A); Xr = np.random.default_rng(1).normal(size=(n, 8)); ks = list(range(0, 41, 2)); spread = []
for k in ks:
    Xk = np.linalg.matrix_power(Ah, k) @ Xr; Xk = Xk / (np.linalg.norm(Xk) + 1e-12); D_ = np.linalg.norm(Xk[:, None] - Xk[None], axis=2).mean(); spread.append(D_)
plt.semilogy(ks, np.array(spread) / spread[0]); plt.xlabel("propagation steps k"); plt.ylabel("mean pairwise distance (normalised, relative to k=0)"); plt.title("Over-smoothing: node representations collapse"); show("oversmoothing")
check(spread[-1] < .5 * spread[0], "after 40 propagation steps node representations are much more similar (over-smoothed)")
# with and without a residual connection (alpha restart, like APPNP / PageRank)
alpha = .1; Xs, Xp = Xr.copy(), Xr.copy()
for _ in range(40): Xs = Ah @ Xs; Xp = (1 - alpha) * Ah @ Xp + alpha * Xr
sp = lambda Z: np.linalg.norm((Z / np.linalg.norm(Z))[:, None] - (Z / np.linalg.norm(Z))[None], axis=2).mean()
print(f"after 40 steps: plain {sp(Xs):.4f}   with restart/residual {sp(Xp):.4f}"); check(sp(Xp) > sp(Xs), "a residual/restart term keeps representations distinguishable"); METRICS["oversmoothing_ratio"] = spread[-1] / spread[0]

In [ ]:
save_metrics("21", METRICS)
log.info('notebook 21 finished OK')

## Wrap-up

## Exercises (do these next)

1. Add edge-dropout (DropEdge) during GCN training and measure test accuracy.
2. Implement a sparse-adjacency GCN with `torch.sparse` and verify equality with the dense one.
3. Build a graph-level classifier (cycles vs trees) with a GIN + sum read-out on synthetic graphs.

## Where to read more

- Wiki chapter: [Module 21](https://github.com/anuxlab/deep-learning-systems/wiki/Module-21) (derivations, worked example, Q&A).
- Kipf & Welling (2017) GCN; Hamilton et al. (2017) GraphSAGE; Veličković et al. (2018) GAT; Xu et al. (2019) *How powerful are GNNs?*; Zachary (1977); Bronstein et al. (2021) *Geometric deep learning*.